In [4]:
import re
from datetime import date
from database import get_conn
from helpers import print_header, print_line


def _valid_email(email):
    return re.match(r"^[^\s@]+@[^\s@]+\.[^\s@]+$", email) is not None


def register():
    try:
        print_header("REGISTER")
        name = input("Full Name: ").strip()
        if len(name) < 2:
            print("  Name too short.")
            return

        email = input("Email: ").strip()
        if not _valid_email(email):
            print("  Invalid email format.")
            return

        password = input("Password (min 6 chars): ").strip()
        if len(password) < 6:
            print("  Password must be at least 6 characters.")
            return

        confirm_pw = input("Confirm Password: ").strip()
        if confirm_pw != password:
            print("  Passwords do not match.")
            return

        print_line()
        print("Choose account type:")
        print("  1. Buyer")
        print("  2. Seller")
        choice = input("Enter (1 or 2): ").strip()
        if choice == "1":
            role = "user"
            approved = 1
        elif choice == "2":
            role = "seller"
            approved = 0   # Sellers need admin approval
        else:
            print("  Invalid choice.")
            return

        conn = get_conn()
        c = conn.cursor()
        c.execute("SELECT id FROM users WHERE email = ?", (email,))
        if c.fetchone():
            print("  Email is already registered.")
            conn.close()
            return

        c.execute(
            "INSERT INTO users (name, email, password, role, approved, joined) VALUES (?, ?, ?, ?, ?, ?)",
            (name, email, password, role, approved, date.today().isoformat())
        )
        conn.commit()
        conn.close()

        if role == "seller":
            print("\n  Seller account created. Awaiting admin approval.")
        else:
            print(f"\n  Welcome {name}! You can now log in.")
    except Exception as e:
        print(" Error in Registeration")
        


def login():

    print_header("LOGIN")
    try:
        email = input("Email: ").strip()
        password = input("Password: ").strip()

        conn = get_conn()
        c = conn.cursor()
        c.execute("SELECT * FROM users WHERE email = ?", (email,))
        user = c.fetchone()
        conn.close()

        if not user:
            print("  No account with this email.")
            return None
        if user["password"] != password:
            print("  Incorrect password.")
            return None
        if user["blocked"]:
            print("  Your account is blocked. Contact admin.")
            return None
        if user["role"] == "seller" and not user["approved"]:
            print("  Your seller account is awaiting admin approval.")
            return None

        print(f"\n  Welcome back, {user['name']}!")
        return dict(user)
    except Exception as e:
        print(" Error in Login:", e)

ModuleNotFoundError: No module named 'database'